*Karen: **[Modeling owner]**: extracted from `airbnb_pricing_crispdm_v7.ipynb` for individual CRISP-DM phase attribution. Run from the repo root (same folder as `db.py`, `pricing_model.py`, etc.).*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install xgboost --quiet
from pricing_model import train_all_markets
results = train_all_markets()


## 4. Modeling

Three model families — `RandomForestRegressor`, `GradientBoostingRegressor`, and `XGBRegressor` — are
tuned **independently per market** via `RandomizedSearchCV`, not one fixed algorithm across all 10
markets. Whichever family scores best on cross-validation R² (train-split only) is what actually gets
saved and served for that market. This replaced the original single-Random-Forest-everywhere approach
after a notebook comparison (see Evaluation, below) showed Random Forest wasn't uniformly best — it
won only 2 of 10 markets when tested against the alternatives.

The feature set is unchanged: `accommodates`, `bedrooms`, `bathrooms` (approximated from `bedrooms`),
`dist_to_center_km`, `host_is_superhost`, `review_scores_rating`, `num_reviews`, `property_type`,
log-transformed `minimum_nights`/`maximum_nights`, `instant_bookable`, `host_response_rate`,
`host_acceptance_rate`, `host_total_listings_count`, all six review sub-scores, `amenities_count`, and
`neighbourhood` (K-fold target-encoded, out-of-fold on the training split only, so a listing's own
price never leaks into its own encoded feature). `room_type` and `property_type` are one-hot encoded,
scoped to each market's own categories.

Model selection uses each candidate's **CV score**, not the held-out test set — choosing a model family
based on test performance would leak test information into the decision, the same double-dipping the
project already avoids for hyperparameter tuning. The test set is touched exactly once, after the
winner is picked, purely to report honest final metrics.


## 5. Evaluation

Each per-market model is evaluated on a held-out 20% split *within that market*, using MAE and R2.
The table below is produced directly by the training run above.

In [ ]:
# XGBoost needs to be installed BEFORE pricing_model.py is imported below - it checks
# for xgboost at import time (HAS_XGB), so installing it after import won't be picked up
# without restarting the kernel.
!pip install xgboost --quiet


In [ ]:
from pricing_model import train_all_markets

results = train_all_markets()

In [ ]:
eval_df = pd.DataFrame(results).T
eval_df